# Part 7: Training and Text Generation

## The Grand Finale!

In this notebook, we'll:
1. Load our PyTorch GPT implementation
2. Train on Shakespeare text
3. Generate new text in Shakespeare's style
4. Experiment with generation parameters

---


In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import os

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# For reproducibility
torch.manual_seed(42)


## Step 1: Load and Explore the Data


In [ ]:
# Load Shakespeare data
data_path = '../data/sample_text.txt'

with open(data_path, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Total characters: {len(text):,}")
print(f"\nFirst 500 characters:")
print("=" * 50)
print(text[:500])
print("=" * 50)


## Step 2: Create the Tokenizer


In [ ]:
class CharTokenizer:
    """Simple character-level tokenizer."""
    
    def __init__(self, text):
        chars = sorted(set(text))
        self.char_to_idx = {c: i for i, c in enumerate(chars)}
        self.idx_to_char = {i: c for i, c in enumerate(chars)}
        self.vocab_size = len(chars)
    
    def encode(self, text):
        return [self.char_to_idx[c] for c in text if c in self.char_to_idx]
    
    def decode(self, indices):
        return ''.join(self.idx_to_char.get(i, '?') for i in indices)

# Create tokenizer
tokenizer = CharTokenizer(text)
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"\nVocabulary:")
print(''.join(tokenizer.char_to_idx.keys()))

# Test encoding/decoding
test_text = "Hello, World!"
encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded)
print(f"\nTest: '{test_text}' -> {encoded[:10]}... -> '{decoded}'")


## Step 3: Create Dataset and DataLoader


In [ ]:
class TextDataset(Dataset):
    """Dataset for autoregressive language modeling."""
    
    def __init__(self, text, tokenizer, seq_len):
        self.seq_len = seq_len
        self.data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
    
    def __len__(self):
        return max(0, len(self.data) - self.seq_len)
    
    def __getitem__(self, idx):
        x = self.data[idx:idx + self.seq_len]
        y = self.data[idx + 1:idx + self.seq_len + 1]
        return x, y

# Create dataset
seq_len = 128
batch_size = 32

dataset = TextDataset(text, tokenizer, seq_len)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"Sequence length: {seq_len}")
print(f"Dataset size: {len(dataset):,} samples")
print(f"Batches per epoch: {len(dataloader)}")

# Show example batch
x_sample, y_sample = next(iter(dataloader))
print(f"\nSample batch shapes:")
print(f"  Input (x): {x_sample.shape}")
print(f"  Target (y): {y_sample.shape}")


## Step 4: Build the GPT Model

We'll use our PyTorch implementation from the `src/` folder:


In [ ]:
# Import our GPT implementation
from src.gpt import GPT, create_gpt_small

# Create model
model = create_gpt_small(tokenizer.vocab_size, max_seq_len=seq_len)
model = model.to(device)

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"Model created!")
print(f"Total parameters: {num_params:,}")
print(f"\nModel architecture:")
print(model)


## Step 5: Training Loop


In [ ]:
def train_epoch(model, dataloader, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        
        # Forward pass
        logits, loss = model(x, y)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)


@torch.no_grad()
def generate_text(model, tokenizer, prompt, max_tokens=100, temperature=0.8, top_k=40):
    """Generate text from a prompt."""
    model.eval()
    
    input_ids = torch.tensor([tokenizer.encode(prompt)], device=device)
    
    for _ in range(max_tokens):
        # Crop to max seq len
        input_crop = input_ids[:, -seq_len:]
        
        # Get predictions
        logits, _ = model(input_crop)
        logits = logits[:, -1, :] / temperature
        
        # Top-k filtering
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')
        
        # Sample
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_token], dim=1)
    
    return tokenizer.decode(input_ids[0].tolist())


In [ ]:
# Training configuration
epochs = 30  # Increase for better results
lr = 3e-4

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

# Training history
losses = []

print("Starting training...")
print("=" * 50)

for epoch in range(epochs):
    loss = train_epoch(model, dataloader, optimizer, device)
    losses.append(loss)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | Loss: {loss:.4f}")
        
        # Generate sample
        sample = generate_text(model, tokenizer, "ROMEO\n", max_tokens=100)
        print(f"Sample: {sample[:150]}...")
        print("-" * 50)

print("\nTraining complete!")


## Step 6: Visualize Training Progress


In [ ]:
# Plot training loss
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(losses, lw=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss Over Time')
ax.grid(True, alpha=0.3)
plt.show()

print(f"Final loss: {losses[-1]:.4f}")
print(f"Loss reduction: {(losses[0] - losses[-1]) / losses[0] * 100:.1f}%")


## Step 7: Generate Text!


In [ ]:
# Generate with different prompts
prompts = [
    "ROMEO\n",
    "To be or not to be",
    "The ",
    "JULIET\nO Romeo, "
]

print("Text Generation Results")
print("=" * 60)

for prompt in prompts:
    print(f"\nPrompt: {repr(prompt)}")
    print("-" * 40)
    generated = generate_text(model, tokenizer, prompt, max_tokens=200, temperature=0.8)
    print(generated)
    print("=" * 60)


## Step 8: Experiment with Temperature


In [ ]:
# Compare different temperatures
prompt = "ROMEO\n"
temperatures = [0.3, 0.7, 1.0, 1.5]

print("Effect of Temperature on Generation")
print("=" * 60)

for temp in temperatures:
    print(f"\nTemperature: {temp}")
    print("-" * 40)
    generated = generate_text(model, tokenizer, prompt, max_tokens=150, temperature=temp, top_k=40)
    print(generated)
    print()

print("\nNote:")
print("- Low temperature (0.3): More deterministic, repetitive")
print("- Medium temperature (0.7-1.0): Balanced creativity")
print("- High temperature (1.5): More random, creative but may be incoherent")


## Step 9: Interactive Generation


In [ ]:
# Try your own prompts!
# Modify this cell to experiment with different prompts

my_prompt = "What light through yonder window breaks? "
my_temperature = 0.8
my_max_tokens = 200

print(f"Your prompt: {repr(my_prompt)}")
print(f"Temperature: {my_temperature}")
print(f"Max tokens: {my_max_tokens}")
print("-" * 50)

generated = generate_text(
    model, tokenizer, my_prompt, 
    max_tokens=my_max_tokens, 
    temperature=my_temperature
)
print(generated)


## Congratulations!

You've built a complete Transformer-based language model from scratch!

### What You've Learned

1. **The Evolution**: From RNNs to Attention to Transformers
2. **Embeddings**: Token and positional encodings
3. **Self-Attention**: The core innovation (Q, K, V)
4. **Multi-Head Attention**: Multiple perspectives
5. **Transformer Block**: Attention + FFN + Norms + Residuals
6. **GPT Architecture**: Decoder-only for generation
7. **Training**: Next-token prediction
8. **Generation**: Autoregressive sampling with temperature

### Next Steps

- Train longer for better results
- Try larger models (more layers, bigger dimensions)
- Experiment with different datasets
- Implement beam search for better generation
- Add attention visualization
- Try fine-tuning on specific tasks

### Resources

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) - Original paper
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) - Visual guide
- [GPT-2 Paper](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
- [nanoGPT](https://github.com/karpathy/nanoGPT) - Minimal GPT implementation
